# Staged Inference

This notebook runs inference through the CardioFusion-XAI staged fusion pipeline:

`Biomarkers → ECG → Echo`

The notebook:
- loads the disease registry,
- loads calibrated modality models,
- accepts whichever modalities are available,
- produces disease-specific risk and confidence,
- recommends missing/confirmatory modalities,
- does not blindly average or multiply probabilities,
- keeps unsupported diseases disabled.

In [44]:
# Cell 1 — Locate project root and import project modules

from pathlib import Path
import sys
import json
import yaml
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "ml-service" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "ml-service":
    raise RuntimeError(
        "Could not locate ml-service. "
        "Run this notebook from inside CardioFusion-XAI/ml-service."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.fusion.registry_loader import load_disease_registry
from src.fusion.staged_fusion import StagedFusionEngine

Project root: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service


In [45]:
# Cell 2 — Load disease registry

REGISTRY_PATH = PROJECT_ROOT / "config" / "disease_registry.yaml"

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry not found: {REGISTRY_PATH}")

registry = load_disease_registry(REGISTRY_PATH)

print("Registry:", REGISTRY_PATH)
print("Diseases:", len(registry.get("diseases", {})))

print("\nRegistered diseases:")
for disease_id, spec in registry["diseases"].items():
    print(
        f"- {disease_id}: "
        f"primary={spec.get('primary_modality')} | "
        f"confirmatory={spec.get('confirmatory_modalities', [])} | "
        f"status={spec.get('status', 'unknown')}"
    )

Registry: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\config\disease_registry.yaml
Diseases: 26

Registered diseases:
- acute_mi: primary=biomarkers | confirmatory=['ecg'] | status=supported
- myocardial_ischemia: primary=ecg | confirmatory=[] | status=supported_weaker
- st_t_abnormalities: primary=ecg | confirmatory=[] | status=supported
- atrial_fibrillation: primary=ecg | confirmatory=[] | status=supported
- vt_vf: primary=None | confirmatory=[] | status=unsupported
- bradyarrhythmia: primary=ecg | confirmatory=[] | status=supported
- av_block: primary=ecg | confirmatory=[] | status=supported
- lbbb_rbbb: primary=ecg | confirmatory=[] | status=supported
- other_conduction_abnormalities: primary=ecg | confirmatory=[] | status=supported
- lvh_rvh: primary=ecg | confirmatory=[] | status=supported_weaker
- lv_dysfunction: primary=echo | confirmatory=[] | status=supported
- reduced_lvef: primary=echo | confirmatory=[] | status=supported
- reduced_ef_cardiomyopathy: primar

In [46]:
# Cell 3 — Load available calibration files

CALIBRATION_DIR = PROJECT_ROOT / "checkpoints" / "calibration"

print("Calibration directory:", CALIBRATION_DIR)

if CALIBRATION_DIR.exists():
    calibration_files = sorted(CALIBRATION_DIR.glob("*.pkl"))
    print(f"Found {len(calibration_files)} calibration files:")
    for path in calibration_files:
        print("  -", path.name)
else:
    calibration_files = []
    print("Calibration directory does not exist yet.")

Calibration directory: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\calibration
Found 1 calibration files:
  - biomarker_acute_mi.pkl


In [47]:
# Cell 4 — Prepare inference inputs
#
# Replace these example dictionaries with outputs produced by your
# actual trained modality models.

# Biomarker predictions:
biomarker_outputs = {}

# ECG predictions:
ecg_outputs = {}

# Echo prediction:
echo_outputs = {}

# Example structure:
#
# biomarker_outputs = {
#     "acute_mi": 0.91,
#     "hf_mortality": 0.28,
#     "ten_year_chd_risk": 0.41,
# }
#
# ecg_outputs = {
#     "acute_mi": 0.73,
#     "atrial_fibrillation": 0.88,
#     "av_block": 0.12,
# }
#
# echo_outputs = {
#     "ef": 32.5,
#     "reduced_lvef": 0.93,
#     "reduced_ef_cardiomyopathy": 0.82,
# }

modality_outputs = {
    "biomarkers": biomarker_outputs,
    "ecg": ecg_outputs,
    "echo": echo_outputs,
}

print("Available modality evidence:")
for modality, outputs in modality_outputs.items():
    print(f"{modality}: {len(outputs)} outputs")

Available modality evidence:
biomarkers: 0 outputs
ecg: 0 outputs
echo: 0 outputs


In [48]:
# Cell 5 — Basic input validation

def validate_probability(value, name):
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be between 0 and 1, got {value}")
    return value

def validate_modality_outputs(outputs):
    if not isinstance(outputs, dict):
        raise TypeError("Each modality output must be a dictionary.")

    cleaned = {}

    for key, value in outputs.items():
        if value is None:
            continue

        # Echo EF is a measurement, not a probability.
        if key.lower() in {"ef", "ejection_fraction", "lvef"}:
            cleaned[key] = float(value)
        else:
            cleaned[key] = validate_probability(value, f"{key}")

    return cleaned

modality_outputs = {
    modality: validate_modality_outputs(outputs)
    for modality, outputs in modality_outputs.items()
}

print("Input validation complete.")

Input validation complete.


In [49]:
# Cell 7 - Run staged inference
# Inputs remain empty until real patient modality predictions are supplied.
# The engine still evaluates every registered disease and reports missing evidence.
fusion_engine = StagedFusionEngine(
    registry=registry,
)

results = fusion_engine.evaluate_all(modality_outputs)

results_list = list(results.values()) if isinstance(results, dict) else list(results)
results_df = pd.DataFrame(results_list)

print(f"Generated {len(results_list)} disease results.")

Generated 26 disease results.


In [50]:
# Cell 7 — Run staged inference
#
# The engine follows the registry:
# - unsupported diseases remain unsupported,
# - missing primary modality -> risk=None + recommendation,
# - available primary -> calibrated primary risk,
# - confirmatory agreement -> confidence upgrade,
# - disagreement -> confidence capped at MODERATE,
# - no blind probability averaging/multiplication.

results = fusion_engine.evaluate_all(modality_outputs)

if isinstance(results, dict):
    results_list = list(results.values())
else:
    results_list = list(results)

results_df = pd.DataFrame(results_list)

print(f"Generated {len(results_list)} disease results.")

Generated 26 disease results.


In [51]:
# Cell 9 - Show only actionable findings

actionable = results_df[
    results_df["status"].isin([
        "primary_only",
        "primary_missing",
        "agreement",
        "disagreement",
        "unsupported",
    ])
].copy()

if "risk" in actionable.columns:
    actionable = actionable.sort_values(
        by="risk",
        ascending=False,
        na_position="last"
    )

display(actionable)

,disease,display_name,risk,confidence,status,detected_from,recommendations,message
0,acute_mi,Acute Myocardial Infarction,None,LOW,primary_missing,[],[Upload biomarkers.],NaN
1,myocardial_ischemia,Myocardial Ischemia,None,LOW,primary_missing,[],[Upload ecg.],NaN
2,st_t_abnormalities,ST/T Abnormalities,None,LOW,primary_missing,[],[Upload ecg.],NaN
3,atrial_fibrillation,Atrial Fibrillation,None,LOW,primary_missing,[],[Upload ecg.],NaN
4,vt_vf,Ventricular Tachycardia / Ventricular Fibrilla...,None,LOW,unsupported,[],[],No sufficient general-population VT/VF ground ...
5,bradyarrhythmia,Bradyarrhythmia,None,LOW,primary_missing,[],[Upload ecg.],NaN
6,av_block,Atrioventricular Block,None,LOW,primary_missing,[],[Upload ecg.],NaN
7,lbbb_rbbb,Left / Right Bundle Branch Block,None,LOW,primary_missing,[],[Upload ecg.],NaN
8,other_conduction_abnormalities,Other Conduction Abnormalities,None,LOW,primary_missing,[],[Upload ecg.],NaN
9,lvh_rvh,Left / Right Ventricular Hypertrophy,None,LOW,primary_missing,[],[Upload ecg.],NaN


In [52]:
# Cell 9 — Show only actionable findings

actionable = results_df[
    results_df["status"].isin([
        "supported",
        "missing_primary",
        "confirmatory_missing",
        "disagreement",
    ])
].copy()

if "risk" in actionable.columns:
    actionable = actionable.sort_values(
        by="risk",
        ascending=False,
        na_position="last"
    )

display(actionable)

,disease,display_name,risk,confidence,status,detected_from,recommendations,message


In [53]:
# Cell 10 — Human-readable staged report

def format_risk(risk):
    if risk is None or pd.isna(risk):
        return "Not available"
    return f"{float(risk) * 100:.1f}%"

print("=" * 80)
print("CARDIOFUSION-XAI — STAGED INFERENCE REPORT")
print("=" * 80)

for _, row in results_df.iterrows():
    print(f"\n{row['display_name']}")

    print(f"  Status      : {row['status']}")
    print(f"  Risk        : {format_risk(row['risk'])}")
    print(f"  Confidence  : {row['confidence']}")
    print(f"  Detected by : {row.get('detected_from', [])}")

    recommendations = row.get("recommendations", [])
    if isinstance(recommendations, str):
        recommendations = [recommendations]

    for recommendation in recommendations:
        print(f"  Recommendation: {recommendation}")

CARDIOFUSION-XAI — STAGED INFERENCE REPORT

Acute Myocardial Infarction
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload biomarkers.

Myocardial Ischemia
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

ST/T Abnormalities
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

Atrial Fibrillation
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

Ventricular Tachycardia / Ventricular Fibrillation
  Status      : unsupported
  Risk        : Not available
  Confidence  : LOW
  Detected by : []

Bradyarrhythmia
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

Atrioventricular Block
  Status      

In [54]:
# Cell 11 — Save inference results

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "staged_inference_results.csv"
json_path = OUTPUT_DIR / "staged_inference_results.json"

results_df.to_csv(csv_path, index=False)

with json_path.open("w", encoding="utf-8") as f:
    json.dump(results_list, f, indent=2, default=str)

print("Saved:")
print(" -", csv_path)
print(" -", json_path)

Saved:
 - d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\inference\staged_inference_results.csv
 - d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\inference\staged_inference_results.json


In [55]:
# Cell 12 — Validate output integrity and print the current staged report

import importlib
import src.fusion.staged_fusion as staged_fusion_module

staged_fusion_module = importlib.reload(
    staged_fusion_module
)
StagedFusionEngine = staged_fusion_module.StagedFusionEngine

fusion_engine = StagedFusionEngine(
    registry=registry,
)

validated_modality_outputs = {
    modality: validate_modality_outputs(outputs)
    for modality, outputs in modality_outputs.items()
}

results = fusion_engine.evaluate_all(
    validated_modality_outputs
)

expected_diseases = set(
    registry.get("diseases", {})
)
actual_results = set(results)
unexpected_results = actual_results - expected_diseases
missing_results = expected_diseases - actual_results

print("Registry diseases:", len(expected_diseases))
print("Inference results:", len(actual_results))
print("Available evidence:")
for modality, outputs in validated_modality_outputs.items():
    print(f"  {modality}: {len(outputs)} prediction fields")

if unexpected_results:
    raise AssertionError(
        "Inference returned registry sections instead of disease IDs: "
        f"{sorted(unexpected_results)}"
    )

if missing_results:
    raise AssertionError(
        "Inference omitted registered diseases: "
        f"{sorted(missing_results)}"
    )

if not any(validated_modality_outputs.values()):
    print(
        "No modality predictions were supplied. "
        "All supported diseases should have risk=None and status=primary_missing; "
        "unsupported diseases should remain disabled."
    )

results_list = list(results.values())
results_df = pd.DataFrame(results_list)

print("\nCARDIOFUSION-XAI — STAGED INFERENCE REPORT")
print("=" * 72)

for result in results_list:
    risk = result.get("risk")
    risk_text = "Not available" if risk is None else f"{float(risk) * 100:.1f}%"
    print(f"\n{result['display_name']}")
    print(f"  Status      : {result['status']}")
    print(f"  Risk        : {risk_text}")
    print(f"  Confidence  : {result['confidence']}")
    print(f"  Detected by : {result.get('detected_from', [])}")
    recommendations = result.get("recommendations", [])
    for recommendation in recommendations:
        print(f"  Recommendation: {recommendation}")

Registry diseases: 26
Inference results: 26
Available evidence:
  biomarkers: 0 prediction fields
  ecg: 0 prediction fields
  echo: 0 prediction fields
No modality predictions were supplied. All supported diseases should have risk=None and status=primary_missing; unsupported diseases should remain disabled.

CARDIOFUSION-XAI — STAGED INFERENCE REPORT

Acute Myocardial Infarction
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload biomarkers.

Myocardial Ischemia
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

ST/T Abnormalities
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

Atrial Fibrillation
  Status      : primary_missing
  Risk        : Not available
  Confidence  : LOW
  Detected by : []
  Recommendation: Upload ecg.

Ventricular Tachycardia